# Train GNN-LSTM On Exported Graph Data

This notebook mirrors `scripts/train_gnn_lstm.py` in a step-by-step format. It loads the graph dataset exported from recorded bags, trains the GNN-LSTM predictor, compares it against a constant-velocity baseline, and saves the checkpoint and summary JSON used by the live GNN runner.


## Imports And Reusable Helpers

This section brings in the graph model components and defines the same utility helpers used by the training script.


In [ ]:
from pathlib import Path
import sys
import json
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebook' else cwd
scripts_dir = project_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from graph_predictor import (
    GNNLSTM,
    GraphSequenceDataset,
    constant_velocity_predict,
    load_runs,
    trajectory_metrics,
)


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def collate_graph_samples(batch):
    node_seq = torch.stack([sample.node_seq for sample in batch], dim=0)
    edge_seq = torch.stack([sample.edge_seq for sample in batch], dim=0)
    target_future = torch.stack([sample.target_future for sample in batch], dim=0)
    past_xy = torch.stack([sample.past_xy for sample in batch], dim=0)
    return node_seq, edge_seq, target_future, past_xy


def evaluate_model(model, loader, device):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for node_seq, edge_seq, target_future, _past_xy in loader:
            node_seq = node_seq.to(device)
            edge_seq = edge_seq.to(device)
            target_future = target_future.to(device)
            pred = model(node_seq, edge_seq)
            preds.append(pred.cpu())
            targets.append(target_future.cpu())
    pred = torch.cat(preds, dim=0)
    target = torch.cat(targets, dim=0)
    return trajectory_metrics(pred, target)


def evaluate_cv(loader):
    preds = []
    targets = []
    with torch.no_grad():
        for _node_seq, _edge_seq, target_future, past_xy in loader:
            pred = constant_velocity_predict(past_xy, target_future.size(1))
            preds.append(pred)
            targets.append(target_future)
    pred = torch.cat(preds, dim=0)
    target = torch.cat(targets, dim=0)
    return trajectory_metrics(pred, target)


## Experiment Configuration

These values match the defaults from the training script. You can adjust them here interactively for small experiments while keeping the `.py` version as the reproducible baseline.


In [ ]:
GRAPH_ROOT = Path.home() / 'Documents/Thesis/graph_dataset'
OUTPUT_DIR = Path.home() / 'Documents/Thesis/models'
MODEL_NAME = 'gnn_lstm_graph_done.pt'
SUMMARY_NAME = 'summary_gnn_graph_done.json'

EGO_NODE = 'husky_local'
PAST_LEN = 10
FUTURE_LEN = 20
EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3
HIDDEN_DIM = 96
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
SEED = 42
FORCE_CPU = False


## Dataset Loading And Split

This section reads all exported graph runs, creates sliding-window samples, and splits them into train/validation/test subsets.


In [ ]:
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() and not FORCE_CPU else 'cpu')
print('Device:', device)

runs = load_runs(GRAPH_ROOT)
dataset = GraphSequenceDataset(
    runs=runs,
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    ego_node=EGO_NODE,
)

total = len(dataset)
train_len = max(1, int(total * TRAIN_RATIO))
val_len = max(1, int(total * VAL_RATIO))
test_len = total - train_len - val_len
if test_len < 1:
    test_len = 1
    if train_len > val_len:
        train_len -= 1
    else:
        val_len -= 1

generator = torch.Generator().manual_seed(SEED)
train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len], generator=generator
)

print({
    'runs': len(runs),
    'samples': total,
    'train': len(train_set),
    'val': len(val_set),
    'test': len(test_set),
})


In [ ]:
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_graph_samples,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_graph_samples,
)
test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_graph_samples,
)


## Model Definition

This is the same GNN-LSTM configuration used by the training script. The graph encoder handles multi-agent interaction, while the LSTM models temporal evolution.


In [ ]:
cfg = {
    'node_dim': 14,
    'edge_dim': 4,
    'hidden_dim': HIDDEN_DIM,
    'lstm_hidden': LSTM_HIDDEN,
    'lstm_layers': LSTM_LAYERS,
    'future_len': FUTURE_LEN,
    'ego_idx': 0,
    'msg_passes': MSG_PASSES,
    'dropout': DROPOUT,
}
model = GNNLSTM(**cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
model


## Training Loop

Run this cell to train the model. It keeps the best validation checkpoint in memory and prints epoch-wise training progress.


In [ ]:
best_val = float('inf')
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for node_seq, edge_seq, target_future, _past_xy in train_loader:
        node_seq = node_seq.to(device)
        edge_seq = edge_seq.to(device)
        target_future = target_future.to(device)

        optimizer.zero_grad()
        pred = model(node_seq, edge_seq)
        loss = criterion(pred, target_future)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * node_seq.size(0)

    train_loss = running_loss / len(train_set)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for node_seq, edge_seq, target_future, _past_xy in val_loader:
            node_seq = node_seq.to(device)
            edge_seq = edge_seq.to(device)
            target_future = target_future.to(device)
            pred = model(node_seq, edge_seq)
            loss = criterion(pred, target_future)
            val_running += loss.item() * node_seq.size(0)
    val_loss = val_running / len(val_set)

    print(f'Epoch {epoch:02d}/{EPOCHS} train_loss={train_loss:.6f} val_loss={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        best_state = {
            'model_state': model.state_dict(),
            'cfg': {
                **cfg,
                'past_len': PAST_LEN,
                'ego_node': EGO_NODE,
            },
        }

if best_state is None:
    raise RuntimeError('Training did not produce a checkpoint')


## Test Evaluation And Baseline Comparison

This section evaluates the best checkpoint on the test split and compares the GNN-LSTM against a constant-velocity predictor.


In [ ]:
model.load_state_dict(best_state['model_state'])
test_metrics = evaluate_model(model, test_loader, device)
cv_metrics = evaluate_cv(test_loader)
comparison = {
    'constant_velocity': cv_metrics,
    'gnn_lstm': test_metrics,
}
print(json.dumps(comparison, indent=2))


## Save Checkpoint And Summary

The output files written here match the filenames expected by the live GNN simulation runner.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / MODEL_NAME
summary_path = OUTPUT_DIR / SUMMARY_NAME

torch.save(best_state, model_path)

summary = {
    'model': 'gnn_lstm',
    'graph_root': str(GRAPH_ROOT.expanduser()),
    'runs': len(runs),
    'samples': total,
    'train': len(train_set),
    'val': len(val_set),
    'test': len(test_set),
    'comparison': comparison,
    'ADE': test_metrics['ADE'],
    'FDE': test_metrics['FDE'],
    'RMSE': test_metrics['RMSE'],
    'model_path': str(model_path.resolve()),
    'past_len': PAST_LEN,
    'future_len': FUTURE_LEN,
}
summary_path.write_text(json.dumps(summary, indent=2))

print('Saved model to:', model_path)
print('Saved summary to:', summary_path)


## Optional Quick Inspection

Use this section to inspect the saved summary directly after training.


In [ ]:
json.loads(summary_path.read_text())
